In [ ]:
# ============================================================
# CELL 1 — Imports
# ============================================================
# itertools.chain is used in the final plotting cell to flatten
# nested lists of column names into a single iterable so the
# DataFrame can be sliced in one step.
import itertools


In [ ]:
# ============================================================
# CELL 2 — Dependency installation (optional)
# ============================================================
# Uncomment the line below to install the physiomodeler package
# if it is not yet present in your environment.
# physiomodeler provides the Model class used throughout this
# notebook to compose and simulate physiological sub-models.
#!pip install physiomodeler


In [ ]:
# ============================================================
# CELL 3 — Airflow mechanics model  (luchtstromingen_model)
# ============================================================
# This cell models the mechanical behaviour of the respiratory
# system as a two-compartment (airways + alveoli) balloon model
# driven by a pressure-controlled ventilator.
#
# Key relationships:
#   Airway opening → airways → alveoli
#   Flow at each segment = ΔPressure / Resistance  (Ohm's law)
#   Volume change = inflow − outflow (± alveolar gas-exchange flux)
#
# Outputs fed downstream:
#   volume_luchtwegen   (state) — airway volume [L]
#   volume_alveoli      (state) — alveolar volume [L]
#   volume_longen       — total lung volume = airways + alveoli [L]
#   druk_alveoli        — absolute alveolar pressure [cmH2O]
#   druk_luchtwegen     — absolute airway pressure [cmH2O]
#   druk_pleura         — pleural pressure [cmH2O]
#   transpulmonale_druk — transpulmonary pressure [cmH2O]
#   debiet_*            — volumetric flow rates [L/s]
# ============================================================

from physiomodeler import Model
import scipy as sp
import numpy as np

# ------------------------------------------------------------------
# Global scaling factors
# ------------------------------------------------------------------
# elastantie_factor: fraction of alveolar volume that participates
# in elastic recoil.  A value < 1 captures the fact that part of
# the alveolar gas volume is not available for elastic storage
# (e.g. due to inhomogeneous ventilation / shunted alveoli).
elastantie_factor = 0.8

# volume_alv_factor: fraction of alveolar volume used as the
# "effective mixing volume" when computing gas-fraction dynamics.
# This prevents artificially large dilution of inspired gas.
volume_alv_factor = 0.3

# ------------------------------------------------------------------
# Airflow resistance parameters
# ------------------------------------------------------------------
# Both resistances are in cmH2O·s/L (analogous to electrical
# resistance in Ohm's law for gas flow).
parameters_luchtstromingen = {
    # Resistance between the airway opening (ventilator Y-piece)
    # and the central airways — represents ETT + large airways.
    "weerstand_luchtwegopening_luchtwegen": 3.0,

    # Resistance between the central airways and the alveolar space
    # — represents peripheral/small airway resistance.
    "weerstand_luchtwegen_alveoli": 2.0,
}

# ------------------------------------------------------------------
# Non-linear elastic pressure-volume (P-V) curve parameters
# ------------------------------------------------------------------
# Each compartment's elastic recoil pressure is described by a
# 5-parameter function (see elastische_druk below):
#   P(V) = g[0]*(V - g[1])          ← linear (Hookean) term
#         + g[2]*exp(g[3]*V)         ← exponential stiffening at high V
#         + g[4]/V                   ← surface-tension term at low V
#
# Tuples/lists are (g0, g1, g2, g3, g4).
parameters_dynamische_elastantie = {
    # Chest-wall (thorax) P-V parameters.
    # The chest wall acts as a spring that opposes lung collapse
    # at low volumes and expansion at high volumes.
    "g_thoraxwand": (5, 4.6, -240, -2.3, 0),

    # Airway (bronchial) P-V parameters.
    # Airways are relatively stiff; the exponential term captures
    # dynamic airway closure at very low airway volumes.
    "g_luchtwegen": [11, 0.2, 1e-3, 8, -0.06],

    # Alveolar P-V parameters.
    # The exponential stiffening term models the upper inflection
    # point (over-distension); the inverse-volume term models the
    # lower inflection point (surfactant / surface tension).
    "g_alveoli": [2.75, 0.8, 1e-3, 2, -0.24],
}


# ------------------------------------------------------------------
# Ventilator pressure waveform — pressure-controlled ventilation
# ------------------------------------------------------------------
def input_druk_als_blokgolf(time, inputs):
    """
    Generate the airway-opening pressure delivered by a pressure-
    controlled mechanical ventilator as a square (block) wave.

    Parameters
    ----------
    time   : float — current simulation time [s]
    inputs : dict  — ventilator settings

    Returns
    -------
    luchtwegdruk : float — airway opening pressure [cmH2O]

    Pressure waveform:
        │ PIP (PEEP + amplitude)
        │─────┐       ┌─────
        │     │       │
        │ PEEP└───────┘
        └─────────────────> time
          ←→ Ti   ←→ Te
        duty = Ti / (Ti + Te)
    """
    # Breath period in seconds derived from respiratory rate [breaths/min]
    periode = 60 / inputs["ademhalingsfrequentie"]

    # scipy square wave: values oscillate between -1 and +1
    # duty controls the fraction of the period spent at +1 (inspiratory phase)
    blokgolf = sp.signal.square(
        time * 2 * np.pi / periode, duty=inputs["druk_duty"]
    )

    # Rescale the square wave from [-1, +1] to [0, amplitude]
    # then shift up by PEEP so the waveform spans [PEEP, PEEP+amplitude]
    luchtwegdruk = (
        (blokgolf + 1)
        / 2
        * inputs["amplitude_luchtwegdruk"]
    )
    luchtwegdruk += inputs["PEEP"]
    return luchtwegdruk


# ------------------------------------------------------------------
# Ventilator settings (inputs to the airflow model)
# ------------------------------------------------------------------
inputs = {
    # Function that produces the time-varying airway-opening pressure
    "druk_luchtwegopening": input_druk_als_blokgolf,

    # Driving pressure above PEEP (peak inspiratory pressure − PEEP) [cmH2O]
    "amplitude_luchtwegdruk": 5,

    # Respiratory rate [breaths/min]
    "ademhalingsfrequentie": 20,

    # Duty cycle: fraction of each breath spent in inspiration (0–1)
    "druk_duty": 0.5,

    # Positive end-expiratory pressure (PEEP) [cmH2O]
    # Prevents alveolar collapse at end-expiration.
    "PEEP": 5,
}


# ------------------------------------------------------------------
# Non-linear elastic (recoil) pressure function
# ------------------------------------------------------------------
def elastische_druk(volume, g):
    """
    Compute the elastic recoil pressure of a lung compartment as a
    function of its volume using a 5-parameter non-linear model.

    P(V) = g[0]*(V − g[1])   # linear stiffness centred at V = g[1]
          + g[2]*exp(g[3]*V) # exponential term (stiffening / softening)
          + g[4] / V         # surface-tension / inverse-volume term

    Parameters
    ----------
    volume : float — compartment volume [L]
    g      : list  — 5-element parameter vector for that compartment

    Returns
    -------
    float — elastic recoil pressure [cmH2O]
    """
    return (
        g[0] * (volume - g[1])        # Hookean (linear) spring term
        + g[2] * np.exp(g[3] * volume) # exponential stiffening/softening
        + g[4] / volume                # surface-tension / inverse-volume
    )


# ------------------------------------------------------------------
# Dynamic elastance pressure block
# ------------------------------------------------------------------
def drukken_dynamische_elastantie(inputs, parameters):
    """
    Compute all pressure quantities from the current compartment
    volumes using the non-linear P-V relationships.

    Anatomical interpretation:
      P_pleura     = chest-wall recoil pressure
                     (acting on the outside of the lung)
      P_E_alveoli  = alveolar elastic recoil pressure
                     (transpulmonary pressure component)
      P_alveoli    = absolute alveolar gas pressure
                     = P_alveoli_elastic + P_pleura
      P_luchtwegen = absolute airway pressure
                     = P_airway_elastic + P_pleura

    The alveolar elastic recoil is evaluated on the *functional*
    alveolar volume (= V_alv / elastantie_factor) because not all
    alveolar gas volume contributes to elastic storage.
    """
    V_luchtwegen = inputs["volume_luchtwegen"]   # airway volume [L]
    V_alveoli    = inputs["volume_alveoli"]       # alveolar volume [L]
    V_longen     = V_luchtwegen + V_alveoli       # total lung volume [L]

    # Effective alveolar volume for elastic-pressure calculation.
    # Dividing by elastantie_factor (<1) scales the volume UP,
    # representing the fact that a smaller recruited fraction carries
    # the full elastic load.
    functioneel_volume_alveoli = V_alveoli / elastantie_factor

    # Elastic recoil of the central airways
    P_E_luchtwegen = elastische_druk(
        V_luchtwegen, parameters["g_luchtwegen"]
    )
    # Elastic recoil of the alveolar compartment (transpulmonary component)
    P_E_alveoli = elastische_druk(
        functioneel_volume_alveoli, parameters["g_alveoli"]
    )
    # Elastic recoil of the chest wall = pleural pressure
    P_E_thoraxwand = elastische_druk(
        V_longen, parameters["g_thoraxwand"]
    )

    return {
        # Pleural pressure: chest-wall recoil (gauge, cmH2O)
        "druk_pleura": P_E_thoraxwand,

        # Transpulmonary pressure: alveolar elastic recoil
        "transpulmonale_druk": P_E_alveoli,

        # Absolute alveolar pressure: sum of alveolar + pleural recoil
        # This is the pressure that drives flow from alveoli outward.
        "druk_alveoli": P_E_alveoli + P_E_thoraxwand,

        # Airway elastic recoil pressure alone
        "elastische_druk_luchtwegen": P_E_luchtwegen,

        # Absolute airway pressure: sum of airway + pleural recoil
        "druk_luchtwegen": P_E_luchtwegen + P_E_thoraxwand,

        # Total lung volume (convenience output for plotting)
        "volume_longen": V_longen,
    }


# Wrap the pressure block as a stand-alone Model so it can be
# reused as an upstream dependency in the composite system model.
dynamische_elastantie_drukken_model = Model(
    dynamics=drukken_dynamische_elastantie,
    parameters=parameters_dynamische_elastantie,
)


# ------------------------------------------------------------------
# Airflow dynamics (Ohm's law for gas flow)
# ------------------------------------------------------------------
def dynamics(inputs, parameters):
    """Dynamica van een simpel ballon-model van de longen.

    Models gas flow through a two-segment airway tree using the
    pneumatic analogue of Ohm's law:  Q = ΔP / R

    Segment 1: airway opening → central airways
        Q_LWO_LW = (P_airway_opening - P_airways) / R_LWO_LW

    Segment 2: central airways → alveoli
        Q_LW_alv = (P_airways - P_alveoli) / R_LW_alv

    Volume continuity:
        dV_LW  = Q_LWO_LW - Q_LW_alv   (what enters minus what leaves)
        dV_alv = Q_LW_alv - Σflux_gas   (inflow minus net gas-exchange flux)

    The gas-exchange flux term (O2 + CO2) accounts for the fact that
    O2 is absorbed from the alveoli into blood and CO2 is released,
    resulting in a net change in alveolar gas volume.
    """
    # Airway resistances [cmH2O·s/L]
    R_LWO_LW = parameters["weerstand_luchtwegopening_luchtwegen"]
    R_LW_alv = parameters["weerstand_luchtwegen_alveoli"]

    # Pressures computed upstream by dynamische_elastantie_drukken_model
    P_LWO = inputs["druk_luchtwegopening"]  # ventilator set-point [cmH2O]
    P_LW  = inputs["druk_luchtwegen"]        # central airway pressure
    P_alv = inputs["druk_alveoli"]           # alveolar pressure

    # Volumetric flow rates [L/s]  (positive = toward alveoli)
    Q_LWO_LW = (P_LWO - P_LW) / R_LWO_LW   # flow into airways
    Q_LW_alv = (P_LW  - P_alv) / R_LW_alv  # flow into alveoli

    # Rate of change of airway volume [L/s]
    dV_LW = Q_LWO_LW - Q_LW_alv

    # Net gas-exchange flux (O2 + CO2) computed by flux_alveoli_PC_model.
    # A positive O2 flux means O2 moves from alveoli to blood (volume loss).
    # A negative CO2 flux means CO2 enters alveoli from blood (volume gain).
    totale_flux = (
        inputs["flux_O2_alveoli_PC"]
        + inputs["flux_CO2_alveoli_PC"]
    )

    # Rate of change of alveolar volume [L/s]
    dV_alv = Q_LW_alv - totale_flux

    return {
        "dvolume_luchtwegen": dV_LW,            # d/dt airway volume
        "dvolume_alveoli": dV_alv,              # d/dt alveolar volume
        "debiet_luchtwegopening_luchtwegen": Q_LWO_LW,  # flow at segment 1
        "debiet_luchtwegen_alveoli": Q_LW_alv,          # flow at segment 2
    }


# ------------------------------------------------------------------
# Composite airflow model
# ------------------------------------------------------------------
# physiomodeler resolves the dependency chain automatically:
#   1. dynamische_elastantie_drukken_model runs first to produce pressures
#   2. dynamics() uses those pressures to update volume states
luchtstromingen_model = Model(
    dynamics=[
        dynamische_elastantie_drukken_model,  # upstream: pressures from volumes
        dynamics,                              # downstream: volumes from pressures
    ],
    state_components=[
        "volume_luchtwegen",   # airway volume [L]  — integrated over time
        "volume_alveoli",      # alveolar volume [L] — integrated over time
    ],
    inputs=inputs,
    parameters=parameters_luchtstromingen,
)


In [ ]:
# ============================================================
# CELL 4 — Convective gas transport model  (gasstromingen_model)
# ============================================================
# Tracks the mole-fraction (volumetric fraction) of O2 and CO2
# inside the airways and alveoli as functions of time.
#
# Physics: each compartment is modelled as a well-mixed reactor.
#   dF/dt = (F_in - F_comp) * Q_in / V_comp   (when Q_in > 0)
#
# This "washing" equation means that the faster the flow relative
# to the compartment volume, the faster the local fraction
# equilibrates toward the incoming fraction.
#
# In addition, the alveolar fraction is altered by gas exchange
# with blood (flux term from flux_alveoli_PC_model).
#
# State variables (outputs):
#   fractie_O2_luchtwegen   — O2 fraction in airways  [0–1]
#   fractie_O2_alveoli      — O2 fraction in alveoli  [0–1]
#   fractie_CO2_luchtwegen  — CO2 fraction in airways [0–1]
#   fractie_CO2_alveoli     — CO2 fraction in alveoli [0–1]
# ============================================================

from physiomodeler import Model

# Inspired gas composition at the airway opening (ambient air, ~FiO2 21%)
inputs = {
    "fractie_O2_luchtwegopening":  0.1965,   # inspired O2 fraction (≈21%)
    "fractie_CO2_luchtwegopening": 0.0003,   # inspired CO2 fraction (≈0.03%)
}


def fractie_verandering(
    fractie_a, fractie_b, debiet_a_b, volume_b
):
    """
    Rate of change of gas fraction in compartment B due to
    convective flow from compartment A.

    Only active when flow is directed FROM A TO B (debiet_a_b > 0).
    When flow is reversed, this function returns 0 — the reverse
    direction is handled by the caller as a separate term.

    Formula:
        dF_B/dt = (F_A - F_B) * Q_{A→B} / V_B

    Intuition: if F_A > F_B the fraction in B rises; the rate
    scales with Q/V (the "wash-in" time constant is V/Q).

    Parameters
    ----------
    fractie_a  : float — gas fraction upstream (compartment A)
    fractie_b  : float — gas fraction in current compartment (B)
    debiet_a_b : float — volumetric flow A→B [L/s]; must be > 0
    volume_b   : float — volume of compartment B [L]

    Returns
    -------
    float — dF_B/dt contribution from this flow [s⁻¹]
    """
    if debiet_a_b <= 0:
        return 0   # no washing when flow is not in this direction

    return (fractie_a - fractie_b) * (debiet_a_b / volume_b)


def dynamics(state, inputs, parameters):
    """
    Compute the time derivatives of gas fractions in the airways
    and alveolar compartments for a single gas (O2 or CO2).

    The airway fraction changes through two mechanisms:
      1. Wash-in from the airway opening (during inspiration)
      2. Back-flow from alveoli (during expiration)

    The alveolar fraction changes through:
      1. Wash-in from the airways (during inspiration)
      3. Gas exchange with pulmonary capillaries (absorption/secretion)

    Note: dfractie_alv_LW uses a negated flow because
    debiet_luchtwegen_alveoli is defined positive toward alveoli;
    a negative value therefore means flow is from alveoli to airways
    (expiration), which is the condition for alveolar back-flow
    to affect the airway fraction.
    """
    gas = parameters["gas"]   # "O2" or "CO2"

    # Effective alveolar mixing volume (smaller than true V_alv)
    functioneel_volume_alv = inputs["volume_alveoli"] * volume_alv_factor

    # --- Fraction changes in the airway compartment ---

    # Contribution 1: fresh gas washes in from the airway opening
    # (active when debiet_luchtwegopening_luchtwegen > 0, i.e., inspiration)
    dfractie_LWO_LW = fractie_verandering(
        inputs[f"fractie_{gas}_luchtwegopening"],      # upstream fraction (inspired)
        state[f"fractie_{gas}_luchtwegen"],             # current airway fraction
        inputs["debiet_luchtwegopening_luchtwegen"],   # inspiratory flow
        inputs["volume_luchtwegen"],                   # airway volume
    )

    # Contribution 2: expired alveolar gas back-flows into airways
    # (active when debiet_luchtwegen_alveoli < 0, i.e., expiration;
    # the sign is flipped so fractie_verandering sees a positive flow)
    dfractie_alv_LW = fractie_verandering(
        state[f"fractie_{gas}_alveoli"],               # upstream = alveoli (expiration)
        state[f"fractie_{gas}_luchtwegen"],             # current airway fraction
        -inputs["debiet_luchtwegen_alveoli"],           # expiratory flow (negated)
        inputs["volume_luchtwegen"],
    )

    # Contribution 3: airway gas flows into alveoli during inspiration
    # (active when debiet_luchtwegen_alveoli > 0)
    dfractie_LW_alv = fractie_verandering(
        state[f"fractie_{gas}_luchtwegen"],             # upstream = airways
        state[f"fractie_{gas}_alveoli"],                # current alveolar fraction
        inputs["debiet_luchtwegen_alveoli"],            # inspiratory flow
        functioneel_volume_alv,
    )

    # Net airway fraction derivative: fresh gas in + expired gas back-flow
    dfractie_LW = dfractie_LWO_LW + dfractie_alv_LW

    # Net alveolar fraction derivative:
    #   inspired gas inflow − net gas-exchange flux (from flux model)
    # The flux is divided by the effective alveolar volume to convert
    # [L/s] to [fraction/s].
    dfractie_alv = (
        dfractie_LW_alv
        - inputs[f"flux_{gas}_alveoli_PC"]
        / functioneel_volume_alv
    )

    return {
        f"dfractie_{gas}_luchtwegen": dfractie_LW,   # d/dt airway fraction
        f"dfractie_{gas}_alveoli":    dfractie_alv,  # d/dt alveolar fraction
    }


# ------------------------------------------------------------------
# Separate Model instances for O2 and CO2
# ------------------------------------------------------------------
# The same dynamics() function is reused; the parameter {"gas": ...}
# selects which gas is being simulated.

gasstroming_O2_model = Model(
    dynamics=dynamics,
    state_components=[
        "fractie_O2_luchtwegen",   # O2 fraction in airways [0–1]
        "fractie_O2_alveoli",      # O2 fraction in alveoli [0–1]
    ],
    inputs=inputs,
    parameters={"gas": "O2"},
)

gasstroming_CO2_model = Model(
    dynamics=dynamics,
    state_components=[
        "fractie_CO2_luchtwegen",  # CO2 fraction in airways [0–1]
        "fractie_CO2_alveoli",     # CO2 fraction in alveoli [0–1]
    ],
    inputs=inputs,
    parameters={"gas": "CO2"},
)

# Composite gas-transport model (O2 + CO2 together)
gasstromingen_model = Model(
    dynamics=[gasstroming_O2_model, gasstroming_CO2_model],
)


In [ ]:
# ============================================================
# CELL 5 — Circulatory perfusion model  (perfusie_model)
# ============================================================
# Models blood-gas content (O2 and CO2) in four circulatory
# compartments arranged in series:
#
#   SV ──cardiac output──► PC ──► SA ──► SC ──► SV
#   (systemic veins)  (pulmonary caps)  (systemic caps)
#
# PC ← O2 from alveoli, CO2 to alveoli  (gas exchange in lungs)
# SC → O2 to tissues, CO2 from tissues  (gas exchange in body)
#
# Each compartment is a well-mixed reservoir ("tank-in-series" model).
# Content [L_gas / L_blood] is the state variable for each gas/compartment.
#
# State variables (8 total: 4 compartments × 2 gases):
#   inhoud_O2_PC, inhoud_O2_SA, inhoud_O2_SC, inhoud_O2_SV
#   inhoud_CO2_PC, inhoud_CO2_SA, inhoud_CO2_SC, inhoud_CO2_SV
# ============================================================

from physiomodeler import Model

# ------------------------------------------------------------------
# Compartment volumes [L]
# ------------------------------------------------------------------
parameters = {
    "volume_PC": 0.1,   # pulmonary capillaries — small, fast turnover
    "volume_SA": 1.1,   # systemic arteries
    "volume_SC": 0.3,   # systemic capillaries — site of tissue gas exchange
    "volume_SV": 3.5,   # systemic veins — largest reservoir
}

# ------------------------------------------------------------------
# Circulatory and metabolic inputs
# ------------------------------------------------------------------
inputs = {
    # Cardiac output [L/s] — 5 L/min converted to L/s
    # This drives convective gas transport through all compartments.
    "cardiac_output": 5 / 60,

    # O2 consumed by tissues [L_O2/s] (positive = O2 leaves blood)
    # 0.25 L/min ≈ 250 mL/min — normal resting oxygen consumption (VO2)
    "flux_O2_SC_weefsels": 0.25 / 60,

    # CO2 produced by tissues [L_CO2/s] (negative = CO2 enters blood)
    # RQ ≈ 0.2/0.25 = 0.8, consistent with a mixed substrate
    "flux_CO2_SC_weefsels": -0.2 / 60,
}


# ------------------------------------------------------------------
# Compartment dynamics — version 1 (content only)
# Overridden below by version 2 which also tracks delivery.
# ------------------------------------------------------------------
def dynamics(state, inputs, parameters):
    """
    (First definition — superseded by the version below.)
    Compute d/dt of gas content in one circulatory compartment.

    Convective transport term:
        dC_B/dt = (C_A - C_B) * CO / V_B
    where CO = cardiac output [L/s] and V_B = compartment volume [L].

    Additional flux terms apply to:
      PC: alveolar gas exchange (flux_gas_alveoli_PC, computed upstream)
      SC: tissue metabolism (flux_gas_SC_weefsels, a constant input)
    """
    gas            = parameters["gas"]
    compartiment_a = parameters["voorgaand_compartiment"]  # upstream compartment
    compartiment_b = parameters["compartiment"]             # current compartment

    inhoud_a     = inputs[f"inhoud_{gas}_{compartiment_a}"]   # upstream content
    inhoud_b     = state[f"inhoud_{gas}_{compartiment_b}"]    # current content
    volume_b     = parameters[f"volume_{compartiment_b}"]
    cardiac_output = inputs["cardiac_output"]

    # Convective mixing: blood entering from upstream dilutes / enriches the
    # current compartment toward the upstream content.
    dinhoud_a_b = (
        (inhoud_a - inhoud_b) * cardiac_output / volume_b
    )

    # Gas-exchange flux specific to each compartment
    if compartiment_b == "PC":
        # Pulmonary capillaries: gas diffuses between alveoli and blood
        # (flux is positive for O2 entering blood, negative for CO2 leaving)
        dinhoud_flux = (
            inputs[f"flux_{gas}_alveoli_PC"] / volume_b
        )
    elif compartiment_b == "SC":
        # Systemic capillaries: tissues consume O2 and produce CO2
        # (flux_SC_weefsels is positive for O2 loss, negative for CO2 gain,
        # so the sign is inverted here to get d/dt of blood content)
        dinhoud_flux = (
            -inputs[f"flux_{gas}_SC_weefsels"] / volume_b
        )
    else:
        # SA and SV: purely convective — no gas exchange
        dinhoud_flux = 0

    return {
        f"dinhoud_{gas}_{compartiment_b}": dinhoud_a_b + dinhoud_flux
    }


# ------------------------------------------------------------------
# Compartment dynamics — version 2 (content + oxygen delivery)
# ------------------------------------------------------------------
# This definition replaces the one above in Python's namespace.
# It adds tracking of gas delivery (content × cardiac_output) as
# a diagnostic output useful for calculating DO2, VO2, etc.
def dynamics(state, inputs, parameters):
    """
    Compute d/dt of gas content in one circulatory compartment,
    and also the instantaneous gas delivery rate [L_gas/s].

    Delivery (e.g. oxygen delivery DO2) is defined as:
        delivery_{gas}_{A}_{B} = C_A * cardiac_output
    i.e. the amount of gas leaving compartment A toward compartment B
    per unit time.
    """
    gas            = parameters["gas"]
    compartiment_a = parameters["voorgaand_compartiment"]
    compartiment_b = parameters["compartiment"]

    inhoud_a     = inputs[f"inhoud_{gas}_{compartiment_a}"]
    inhoud_b     = state[f"inhoud_{gas}_{compartiment_b}"]
    volume_b     = parameters[f"volume_{compartiment_b}"]
    cardiac_output = inputs["cardiac_output"]

    # Convective term (identical to version 1)
    dinhoud_a_b = (
        (inhoud_a - inhoud_b) * cardiac_output / volume_b
    )

    # Gas-exchange flux (identical to version 1)
    if compartiment_b == "PC":
        dinhoud_flux = (
            inputs[f"flux_{gas}_alveoli_PC"] / volume_b
        )
    elif compartiment_b == "SC":
        dinhoud_flux = (
            -inputs[f"flux_{gas}_SC_weefsels"] / volume_b
        )
    else:
        dinhoud_flux = 0

    return {
        f"dinhoud_{gas}_{compartiment_b}": dinhoud_a_b + dinhoud_flux,

        # Gas delivery from compartment A to B [L_gas/s]
        # Useful post-hoc for computing DO2 (O2 delivery) or VCO2.
        f"delivery_{gas}_{compartiment_a}_{compartiment_b}": inhoud_a * cardiac_output,
    }


# ------------------------------------------------------------------
# Build one Model per gas per compartment-transition
# ------------------------------------------------------------------
# The loop creates 8 models:
#   O2:  SV→PC, PC→SA, SA→SC, SC→SV
#   CO2: SV→PC, PC→SA, SA→SC, SC→SV
#
# Each model tracks the content in the *downstream* compartment
# (compartiment_b) as its state variable.
modellen = []
for gas in ("O2", "CO2"):
    for compartiment_a, compartiment_b in (
        ("SV", "PC"),   # venous blood enters pulmonary capillaries
        ("PC", "SA"),   # oxygenated blood enters systemic arteries
        ("SA", "SC"),   # arterial blood enters systemic capillaries
        ("SC", "SV"),   # deoxygenated blood returns to systemic veins
    ):
        model = Model(
            dynamics=dynamics,
            state_components=[
                f"inhoud_{gas}_{compartiment_b}"   # gas content in compartment B
            ],
            parameters={
                "gas":                    gas,
                "compartiment":           compartiment_b,
                "voorgaand_compartiment": compartiment_a,
            },
        )
        modellen.append(model)


# ------------------------------------------------------------------
# Safety event: stop simulation if O2 content goes negative
# ------------------------------------------------------------------
# A negative O2 content is physically impossible; if it occurs the
# ODE integrator has diverged.  Marking this as terminal=True causes
# physiomodeler / scipy to halt the simulation at that point.
def event_concentratie_O2_negatief(state):
    """Event function: returns inhoud_O2_SC; zero-crossing signals trouble."""
    return state["inhoud_O2_SC"]


event_concentratie_O2_negatief.terminal = True  # halt on zero-crossing

# ------------------------------------------------------------------
# Composite perfusion model
# ------------------------------------------------------------------
perfusie_model = Model(
    dynamics=modellen,          # list of 8 compartment sub-models
    inputs=inputs,
    parameters=parameters,      # compartment volumes
    events=[event_concentratie_O2_negatief],  # safety termination
)


In [ ]:
# ============================================================
# CELL 6 — Alveolar-capillary gas exchange model
#          (flux_alveoli_PC_model)
# ============================================================
# Computes the rate of O2 and CO2 transfer between alveolar gas
# and pulmonary-capillary blood using Fick's first law of diffusion:
#
#   flux = D_L * (P_alveoli − P_capillary)
#
# where D_L is the lung diffusing capacity [L/(s·kPa)].
#
# The driving force (ΔP) requires partial pressures in both
# compartments.  Alveolar partial pressures come from gas fractions
# and total alveolar pressure; capillary partial pressures are
# derived from the blood-gas content via dissociation curves.
#
# Outputs fed to other models:
#   flux_O2_alveoli_PC   — O2 flux [L/s], positive = O2 enters blood
#   flux_CO2_alveoli_PC  — CO2 flux [L/s], negative = CO2 enters blood
#   partiele_druk_O2_PC  — PO2 in pulmonary capillaries [kPa]
#   saturatie_O2_*       — O2 saturation in each compartment [0–1]
# ============================================================

import numpy as np
from physiomodeler import Model

# ------------------------------------------------------------------
# Fixed atmospheric pressure input
# ------------------------------------------------------------------
inputs = {
    # Atmospheric (barometric) pressure expressed in cmH2O.
    # 1033 cmH2O ≈ 101.3 kPa = 1 atm.
    # Alveolar gauge pressures from the mechanics model are
    # added to this to obtain absolute pressures.
    "druk_atmosfeer": 1033,  # cmH2O
}

# ------------------------------------------------------------------
# Gas-exchange parameters
# ------------------------------------------------------------------
parameters = {
    # Unit conversion: 1 cmH2O = 0.0981 kPa
    "kPa_per_cmH2O": 0.0981,

    # Maximum O2 bound per gram of haemoglobin (Hüfner's constant)
    # ≈ 1.34 mL O2/g Hb; here expressed as L/g
    "bindingscapaciteit_Hb_per_gram": 1.35e-3,  # L(O2)/g(Hb)

    # Haemoglobin concentration in whole blood
    "concentratie_Hb": 150,  # g(Hb)/L(blood)

    # O2 diffusing capacity of the alveolar membrane + capillary blood
    # Typical healthy value ≈ 25 mL/(min·mmHg) ≈ 0.0042 L/(s·kPa)
    "diffusiecapaciteit_O2_alveoli_PC": 0.0042,  # L/(s·kPa)

    # CO2 diffusing capacity — roughly 6× higher than O2 due to
    # greater solubility and reactivity of CO2 in blood
    "diffusiecapaciteit_CO2_alveoli_PC": 0.025,   # L/(s·kPa)
}


# ------------------------------------------------------------------
# Partial pressures in airway gas compartments
# ------------------------------------------------------------------
# (First definition — generalised version below supersedes this one)
def partiele_drukken_lucht(inputs):
    """
    (Superseded — see generalised version below.)
    Compute alveolar partial pressures of O2 and CO2 from gauge
    alveolar pressure and gas fractions (Dalton's law).
    """
    P_alv_abs_cmH2O = (
        inputs["druk_alveoli"] + inputs["druk_atmosfeer"]
    )
    P_alv_abs_kPa = (
        P_alv_abs_cmH2O * parameters["kPa_per_cmH2O"]
    )
    return {
        "partiele_druk_O2_alveoli":  P_alv_abs_kPa * inputs["fractie_O2_alveoli"],
        "partiele_druk_CO2_alveoli": P_alv_abs_kPa * inputs["fractie_CO2_alveoli"],
    }


# ------------------------------------------------------------------
# Generalised partial pressures in gas compartments (alveoli + airways)
# ------------------------------------------------------------------
# This version replaces the one above; it computes partial pressures
# for both the alveoli and the central airways.
def partiele_drukken_lucht(inputs):
    """
    Compute O2 and CO2 partial pressures [kPa] in the alveoli
    and central airways using Dalton's law:

        P_gas = P_total_abs * F_gas

    The gauge pressures (from the mechanics model, in cmH2O) are
    converted to absolute kPa by adding atmospheric pressure and
    applying the unit conversion factor.

    Outputs four partial pressures:
        partiele_druk_O2_alveoli
        partiele_druk_CO2_alveoli
        partiele_druk_O2_luchtwegen
        partiele_druk_CO2_luchtwegen
    """
    resultaten = {}
    for compartiment in ["alveoli", "luchtwegen"]:
        # Convert gauge pressure → absolute [cmH2O] → absolute [kPa]
        P_abs_cmH2O = (
            inputs[f"druk_{compartiment}"]
            + inputs["druk_atmosfeer"]
        )
        P_abs_kPa = (
            P_abs_cmH2O * parameters["kPa_per_cmH2O"]
        )
        # Dalton's law: partial pressure = total × mole fraction
        resultaten.update(
            {
                f"partiele_druk_O2_{compartiment}":  P_abs_kPa * inputs[f"fractie_O2_{compartiment}"],
                f"partiele_druk_CO2_{compartiment}": P_abs_kPa * inputs[f"fractie_CO2_{compartiment}"],
            }
        )
    return resultaten


# ------------------------------------------------------------------
# Inverse O2-Hb dissociation curve  (saturation → PO2)
# ------------------------------------------------------------------
def partiele_druk_O2_bij_saturatie(saturatie):
    """
    Compute the O2 partial pressure [kPa] corresponding to a given
    haemoglobin oxygen saturation using an analytical inverse of the
    Hill-type dissociation curve.

    The standard form is:  S = P^n / (P50^n + P^n)
    Here a parameterised form is solved analytically via the cubic
    formula after reformulating as a depressed cubic:
        x^3 + h2*x − y_N = 0   (where x = a + b is split for Cardano)

    h2 = 2.81 is related to the Hill coefficient and P50;
    y_N is the transformed saturation variable.

    Negative bases are handled carefully using sign(x)*|x|^(1/3)
    to avoid complex numbers in numpy's power operator.

    Parameters
    ----------
    saturatie : float — fractional O2 saturation (0–1)

    Returns
    -------
    float — O2 partial pressure [kPa]
    """
    h2  = 2.81
    y_N = 55.5 * saturatie / (1 - saturatie)  # transformed saturation

    # Cardano split: a + b = PO2
    a = (y_N + np.sqrt(y_N**2 + h2)) / 2
    b = (y_N - np.sqrt(y_N**2 + h2)) / 2

    # Raising a negative number to a fractional power yields complex
    # results in Python.  We preserve the sign explicitly:
    #   np.sign(x) * |x|^(1/3)
    return np.sign(a) * np.abs(a) ** (1 / 3) + np.sign(
        b
    ) * np.abs(b) ** (1 / 3)


# ------------------------------------------------------------------
# CO2 dissociation curve  (content → PCO2)
# ------------------------------------------------------------------
def partiele_druk_CO2_bij_inhoud(inhoud):
    """
    Empirical exponential CO2 dissociation curve.

    Converts total blood CO2 content [L_CO2 / L_blood] to
    partial pressure [kPa].  The exponential form captures the
    non-linear (but relatively linear in physiological range)
    CO2 dissociation characteristic:

        PCO2 [kPa] = 0.837 * exp(4.16 * C) − 0.895

    Parameters
    ----------
    inhoud : float — CO2 content [L_CO2/L_blood]

    Returns
    -------
    float — CO2 partial pressure [kPa]
    """
    return 0.837 * np.exp(4.16 * inhoud) - 0.895


# ------------------------------------------------------------------
# Blood partial pressures (all four circulatory compartments)
# ------------------------------------------------------------------
def partiele_drukken_bloed(inputs, parameters):
    """
    Compute O2 saturation and partial pressures for O2 and CO2
    in all four blood compartments (PC, SA, SC, SV).

    Step 1 — O2 saturation:
        SatO2 = C_O2 / C_Hb_max
        where C_Hb_max = [Hb] * binding_capacity_per_gram
        (assumes dissolved O2 is negligible; Hb-bound dominates)

    Step 2 — O2 partial pressure:
        PO2 = inverse_dissociation_curve(SatO2)

    Step 3 — CO2 partial pressure:
        PCO2 = empirical_CO2_curve(C_CO2)

    Outputs (12 values for 4 compartments × 3 quantities):
        saturatie_O2_{PC,SA,SC,SV}
        partiele_druk_O2_{PC,SA,SC,SV}
        partiele_druk_CO2_{PC,SA,SC,SV}
    """
    concentratie_Hb       = parameters["concentratie_Hb"]              # g/L
    bindingscapaciteit_O2_Hb = parameters["bindingscapaciteit_Hb_per_gram"]  # L/g

    # Total O2-binding capacity of haemoglobin [L_O2/L_blood]
    totale_bindingscapaciteit_O2 = (
        concentratie_Hb * bindingscapaciteit_O2_Hb
    )

    resultaten = {}
    for compartiment in ["PC", "SA", "SC", "SV"]:
        # Step 1: fractional O2 saturation
        resultaten[f"saturatie_O2_{compartiment}"] = (
            inputs[f"inhoud_O2_{compartiment}"]
            / totale_bindingscapaciteit_O2
        )

        # Step 2: PO2 via inverse dissociation curve
        resultaten[f"partiele_druk_O2_{compartiment}"] = (
            partiele_druk_O2_bij_saturatie(
                resultaten[f"saturatie_O2_{compartiment}"]
            )
        )

        # Step 3: PCO2 via empirical dissociation curve
        resultaten[f"partiele_druk_CO2_{compartiment}"] = (
            partiele_druk_CO2_bij_inhoud(
                inputs[f"inhoud_CO2_{compartiment}"]
            )
        )

    return resultaten


# ------------------------------------------------------------------
# Fick diffusion flux across the alveolar-capillary membrane
# ------------------------------------------------------------------
def flux_alveoli_PC(parameters, inputs):
    """
    Compute the net gas flux [L/s] between alveoli and pulmonary
    capillaries using Fick's first law:

        flux = D_L * (P_alveoli − P_capillary)

    For O2:
        P_alveoli  > P_capillary  → positive flux (O2 enters blood)
    For CO2:
        P_capillary > P_alveoli   → negative flux (CO2 enters alveoli)

    The diffusing capacity D_L [L/(s·kPa)] is the product of the
    membrane diffusing capacity and the capillary reaction rate;
    it is much higher for CO2 than O2.

    Parameters
    ----------
    parameters : dict — must contain 'gas' and diffusing capacities
    inputs     : dict — must contain alveolar and capillary partial pressures

    Returns
    -------
    dict — {flux_{gas}_alveoli_PC: float [L/s]}
    """
    gas = parameters["gas"]

    diffusiecapaciteit = parameters[
        f"diffusiecapaciteit_{gas}_alveoli_PC"
    ]

    partiele_druk_alveoli = inputs[
        f"partiele_druk_{gas}_alveoli"
    ]
    partiele_druk_PC = inputs[f"partiele_druk_{gas}_PC"]

    return {
        f"flux_{gas}_alveoli_PC": diffusiecapaciteit
        * (partiele_druk_alveoli - partiele_druk_PC)
    }


# ------------------------------------------------------------------
# Separate gas-exchange models for O2 and CO2
# ------------------------------------------------------------------
flux_O2_alveoli_PC_model = Model(
    dynamics=flux_alveoli_PC,
    parameters={"gas": "O2"},
)

flux_CO2_alveoli_PC_model = Model(
    dynamics=flux_alveoli_PC,
    parameters={"gas": "CO2"},
)

# ------------------------------------------------------------------
# Composite gas-exchange model
# ------------------------------------------------------------------
# Execution order:
#   1. partiele_drukken_lucht  → alveolar/airway partial pressures
#   2. partiele_drukken_bloed  → capillary partial pressures
#   3. flux_O2_alveoli_PC_model → O2 Fick flux
#   4. flux_CO2_alveoli_PC_model → CO2 Fick flux
flux_alveoli_PC_model = Model(
    dynamics=[
        partiele_drukken_lucht,          # Dalton's law for gas phases
        partiele_drukken_bloed,          # dissociation curves for blood
        flux_O2_alveoli_PC_model,        # O2 Fick diffusion
        flux_CO2_alveoli_PC_model,       # CO2 Fick diffusion
    ],
    parameters=parameters,
    inputs=inputs,
)


In [ ]:
# ============================================================
# CELL 7 — Composite system model  (systeem_model)
# ============================================================
# Integrates all five sub-models into a single closed-loop
# respiratory-circulatory system.
#
# physiomodeler resolves the inter-model dependency graph and
# determines the evaluation order automatically.  The order
# listed here reflects the physical causality:
#
#  ┌─────────────────────────────────────────────────────────┐
#  │  1. dynamische_elastantie_drukken_model                 │
#  │     Volumes → Pressures (mechanical)                    │
#  │                                                         │
#  │  2. flux_alveoli_PC_model                               │
#  │     Fractions + Blood contents → Gas-exchange fluxes    │
#  │     (requires pressures from step 1 and contents from 5)│
#  │                                                         │
#  │  3. luchtstromingen_model                               │
#  │     Pressures + Fluxes → Volume derivatives             │
#  │                                                         │
#  │  4. gasstromingen_model                                 │
#  │     Flows + Fluxes → Fraction derivatives               │
#  │                                                         │
#  │  5. perfusie_model                                      │
#  │     Cardiac output + Fluxes → Content derivatives       │
#  └─────────────────────────────────────────────────────────┘
#
# All state derivatives are then passed to the ODE integrator
# (RK45 by default in physiomodeler) to advance the simulation.
# ============================================================

systeem_model = Model(
    dynamics=[
        dynamische_elastantie_drukken_model,  # step 1: mechanics → pressures
        flux_alveoli_PC_model,                # step 2: gas exchange fluxes
        luchtstromingen_model,                # step 3: volume dynamics
        gasstromingen_model,                  # step 4: gas-fraction dynamics
        perfusie_model,                       # step 5: blood-gas content dynamics
    ],
)


In [ ]:
# ============================================================
# CELL 8 — Run the simulation
# ============================================================
# Integrates the system model over 60 seconds of simulated time
# starting from physiologically plausible initial conditions.
#
# Initial state rationale:
#   volume_luchtwegen = 0.7 L   — anatomical dead space (~150 mL
#                                  for airways + ETT approximation)
#   volume_alveoli    = 2.8 L   — near functional residual capacity
#                                  (FRC ≈ 2.5–3.5 L in adults)
#   inhoud_O2_*       = 0.15    — O2 content ≈ 0.15 L/L blood,
#                                  corresponding to SaO2 ≈ 75%
#                                  (mixed venous baseline before
#                                   the circulation reaches steady state)
#   inhoud_CO2_*      = default — CO2 initial values not explicitly
#                                  set here; physiomodeler uses 0 or
#                                  a model default for unspecified states
# ============================================================

result = systeem_model.run_simulation(
    time=60,             # simulation duration [s]
    initial_state={
        "volume_luchtwegen": 0.7,   # airway volume at t=0 [L]
        "volume_alveoli":    2.8,   # alveolar volume at t=0 [L]

        # Blood O2 content in all compartments — uniform at start
        # to let the model spin up to its physiological steady state
        "inhoud_O2_PC": 0.15,  # ±saturatie van 75%
        "inhoud_O2_SA": 0.15,
        "inhoud_O2_SC": 0.15,
        "inhoud_O2_SV": 0.15,
    },
)


In [ ]:
# ============================================================
# CELL 9 — Visualisation
# ============================================================
# Plots four groups of simulation outputs in separate subplots.
#
# Column groups:
#   [0] volume_longen                     — total lung volume [L]
#   [1] fractie_O2_alveoli, fractie_CO2   — alveolar gas fractions
#   [2] inhoud_O2_{PC,SA,SC,SV}           — O2 content in blood [L/L]
#   [3] inhoud_CO2_{PC,SA,SC,SV}          — CO2 content in blood [L/L]
#
# itertools.chain(*columns) flattens the nested list into a single
# flat list of column names; this allows passing all names at once
# to result[[...]] for column selection, while subplots=columns
# tells pandas to group them back into the original sub-lists for
# separate axes.
# ============================================================

columns = [
    # Subplot 1: total lung volume (mechanics)
    ["volume_longen"],

    # Subplot 2: alveolar O2 and CO2 fractions (gas transport)
    ["fractie_O2_alveoli", "fractie_CO2_alveoli"],

    # Subplot 3: O2 content in all four blood compartments (perfusion)
    [
        "inhoud_O2_PC",
        "inhoud_O2_SA",
        "inhoud_O2_SC",
        "inhoud_O2_SV",
    ],

    # Subplot 4: CO2 content in all four blood compartments (perfusion)
    [
        "inhoud_CO2_PC",
        "inhoud_CO2_SA",
        "inhoud_CO2_SC",
        "inhoud_CO2_SV",
    ],
]

# Flatten column groups → select all columns at once from the DataFrame,
# then pass the original nested groups to plot() so each group gets
# its own subplot axis.
result[[*itertools.chain(*columns)]].plot(subplots=columns)
